# CART Regression Tree from Scratch in NumPy

**DATA2060 Final Project**

**Group Members: Weijia Wang, Daniel Ding, Tanush Garg, Thomas Lin**

**Public GitHub repository link: https://github.com/weijiawang2003/DATA-2060-Final**

## 1. Overview
Classification and Regression Trees (CART) is a method for learning a binary decision tree from labeled examples. This project uses its regression form to predict a single numerical response. The implementation takes a dense matrix of finite numerical features and a vector of finite targets. It learns a set of threshold rules using squared error as the criterion for evaluating candidate splits (Breiman et al., 1984).

### Representation

Each internal node stores a feature index and a threshold. A row goes to the left child if the selected feature is less than or equal to the threshold, and to the right child otherwise. Following these rules from the root sends each row to exactly one leaf. All rows that reach the same leaf receive the same prediction. If the tree has $M$ leaves, the corresponding regions $R_1,\ldots,R_M$ partition the feature space, giving the piecewise constant predictor

$$
\hat f(\mathbf{x})=\sum_{m=1}^{M}c_m\,\mathbf{1}\{\mathbf{x}\in R_m\}.
$$

The path of a shallow tree gives a short explanation for each prediction. Splits on different features along the same path can also represent an interaction. For example, the tree can use a humidity threshold only on the branch where temperature exceeds a particular value. These properties make a small tree useful when the form of the relationship is unknown and readers need to see how decisions are made (scikit-learn developers, 2026).

### Loss

We measure prediction error with mean squared error (MSE), the average squared difference between the observed and predicted targets. A low training MSE can coexist with a high test MSE when a deep tree fits noise. For a candidate leaf with training row indices $I$, fitting its prediction means minimizing the following function of the constant $c$:

$$
L_I(c)=\frac{1}{|I|}\sum_{i\in I}(y_i-c)^2,\qquad
\frac{dL_I}{dc}=2(c-\bar y_I),\qquad
\frac{d^2L_I}{dc^2}=2.
$$

The positive second derivative gives a unique minimum at $c=\bar y_I$, the mean of the targets in that leaf. For split selection, we use the sum of squared errors (SSE) about the mean, denoted $S(I)$. It equals the node MSE multiplied by the number of rows in the node. This weighting matters when the two children have different sizes. Comparing the combined SSE counts each training row equally, whereas averaging the two child MSE values with equal weights would give the smaller child more influence per row.

### Optimizer

For a feature $j$ and threshold $t$, the left set contains rows with $x_{ij}\le t$ and the right set contains the other rows. After fitting each child's prediction to its own mean, the improvement is

$$
\Delta(j,t)=S(I)-S(I_L(j,t))-S(I_R(j,t)).
$$

The parent SSE is fixed during the search, so the best split minimizes the combined child SSE. Within a feature, the implementation considers midpoints between successive eligible values in sorted order. All thresholds strictly between two consecutive distinct values produce the same partition of the training rows. Repeated feature values stay together, and candidate selection skips adjacent feature values separated by at most $10^{-7}$ after features are converted to 32-bit floats. Splits that leave too few rows on either side are excluded. If two candidates achieve the same score, the search keeps the first one in feature and threshold order.

The implementation computes the same gain using an equivalent expression. Let $n_L$ and $n_R$ be the two child counts. Expanding the parent SSE about the child means cancels the within-child errors and leaves the between-child term

$$
\Delta(j,t)=\frac{n_L n_R}{n_L+n_R}(\bar y_L-\bar y_R)^2.
$$

Once the rows are sorted by a feature, a cumulative sum of the ordered targets gives both child means at each candidate without summing the same rows again. Subtracting the parent mean from the targets before forming this sum leaves the difference of child means unchanged.

**Worked example.** Suppose one feature takes values $1,2,3,4$ and the corresponding targets are $2,4,8,10$. The root predicts $6$ and has an SSE of $40$. A split at $2.5$ gives child means of $3$ and $9$. Each child has an SSE of $2$, so the split reduces SSE by $36$. An input exactly at $2.5$ follows the left branch.

The algorithm repeats this greedy search independently in each child. Each step optimizes the current split, and the chosen split determines which future splits are available to its descendants. Depth and minimum leaf size control the number and granularity of the regions. A deeper tree can recover smaller local patterns, but its structure may change when the training data change slightly. Its leaf predictions also remain constant beyond the observed feature range along a fixed path, which limits extrapolation (scikit-learn developers, 2026).

```text
GROW(rows, depth):
    prediction = mean of targets in rows
    if depth reaches limit, targets are constant, or rows are too few:
        return leaf(prediction)
    find the feature and threshold with minimum legal combined child SSE
    if there is no legal split:
        return leaf(prediction)
    split rows into left and right children
    return node(feature, threshold,
                GROW(left, depth + 1), GROW(right, depth + 1))
```

## 2. Model

`train(X, y)` grows the tree, `predict(X)` returns one prediction per row, and
`loss(X, y)` reports the mean squared error. Each internal node stores the feature
and threshold for its split. Every node stores its mean target, which is the
prediction when that node becomes a leaf.

In [ ]:
import numpy as np


class CARTRegressor:
    """A squared-error regression tree for dense, finite, numerical data."""

    def __init__(self, max_depth=None, min_samples_leaf=1):
        """Configure the tree and initialize its unfitted state.

        max_depth is a nonnegative integer or None for unlimited depth.
        min_samples_leaf is the minimum number of training samples in either
        child of an accepted split. The root has depth zero.
        """
        if max_depth is not None and (
            isinstance(max_depth, (bool, np.bool_))
            or not isinstance(max_depth, (int, np.integer))
            or max_depth < 0
        ):
            raise ValueError("max_depth must be a nonnegative integer or None.")
        if (
            isinstance(min_samples_leaf, (bool, np.bool_))
            or not isinstance(min_samples_leaf, (int, np.integer))
            or min_samples_leaf < 1
        ):
            raise ValueError("min_samples_leaf must be a positive integer.")

        self.max_depth = None if max_depth is None else int(max_depth)
        self.min_samples_leaf = int(min_samples_leaf)
        self.tree_ = None
        self.n_features_in_ = None

    def train(self, X, y):
        """Fit the tree and return self.

        X has shape (n_samples, n_features), with at least one row and column.
        y is a one-dimensional array of n_samples numerical targets.
        Both inputs must be finite and real-valued.
        """
        if np.iscomplexobj(X) or np.iscomplexobj(y):
            raise ValueError("X and y must be real-valued.")
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float64)
        if X.ndim != 2 or not X.shape[0] or not X.shape[1]:
            raise ValueError("X must be a nonempty two-dimensional array.")
        if y.shape != (X.shape[0],):
            raise ValueError("y must contain one target per row of X.")
        if not np.isfinite(X).all() or not np.isfinite(y).all():
            raise ValueError("X and y must contain only finite values.")

        self.tree_ = self._grow_tree(X, y, depth=0)
        self.n_features_in_ = X.shape[1]
        return self

    def _best_split(self, X, y):
        """Find the candidate split with the largest decrease in squared error.

        X and y are the validated feature matrix and target vector at a node.
        Return (feature index, threshold, Boolean mask for the left child)
        or None if no split meets min_samples_leaf.
        """
        n_samples = len(y)
        min_leaf = self.min_samples_leaf
        if n_samples < 2 * min_leaf:
            return None

        candidate_counts = np.arange(min_leaf, n_samples - min_leaf + 1)
        centered_y = y - y.mean()
        best_gain = -np.inf
        best_feature = None
        best_threshold = None

        for feature in range(X.shape[1]):
            order = np.argsort(X[:, feature], kind="stable")
            values = X[order, feature].astype(np.float64)
            lower = values[candidate_counts - 1]
            upper = values[candidate_counts]

            # Ignore tied or nearly tied values in the float32 feature data.
            valid = upper > lower + 1e-7
            if not valid.any():
                continue

            left_count = candidate_counts[valid]
            right_count = n_samples - left_count
            sums = np.cumsum(centered_y[order])
            left_sum = sums[left_count - 1]
            right_sum = sums[-1] - left_sum

            # Parent SSE minus child SSEs equals this between-child sum of squares.
            mean_difference = left_sum / left_count - right_sum / right_count
            gains = left_count / n_samples * right_count * mean_difference**2
            winner = int(np.argmax(gains))

            # A strict comparison makes feature and threshold tie-breaking stable.
            if gains[winner] > best_gain:
                best_gain = gains[winner]
                best_feature = feature
                best_threshold = float(
                    (lower[valid][winner] + upper[valid][winner]) / 2
                )

        if best_feature is None:
            return None

        left_mask = X[:, best_feature].astype(np.float64) <= best_threshold
        return best_feature, best_threshold, left_mask

    def _grow_tree(self, X, y, depth):
        """Build a subtree from the samples at the given depth.

        Return a dictionary with the node's target mean in 'value'. Split
        nodes also store 'feature', 'threshold', and their two child nodes.
        """
        node = {"value": float(y.mean())}
        max_depth_reached = self.max_depth is not None and depth >= self.max_depth
        if max_depth_reached or np.all(y == y[0]):
            return node

        split = self._best_split(X, y)
        if split is None:
            return node

        feature, threshold, left_mask = split
        right_mask = ~left_mask
        node.update(
            feature=feature,
            threshold=threshold,
            left=self._grow_tree(X[left_mask], y[left_mask], depth + 1),
            right=self._grow_tree(X[right_mask], y[right_mask], depth + 1),
        )
        return node

    def predict(self, X):
        """Return a one-dimensional array of leaf means for the rows of X.

        X must have the same number of features as the training data. Values
        at or below a node's threshold follow its left branch.
        """
        if self.tree_ is None:
            raise RuntimeError("Call train before predict.")
        if np.iscomplexobj(X):
            raise ValueError("X must be real-valued.")
        X = np.asarray(X, dtype=np.float32)
        if X.ndim != 2 or X.shape[1] != self.n_features_in_:
            raise ValueError("X must be two-dimensional with the training features.")
        if not np.isfinite(X).all():
            raise ValueError("X must contain only finite values.")

        predictions = np.empty(X.shape[0], dtype=np.float64)
        for i, row in enumerate(X):
            node = self.tree_
            while "feature" in node:
                feature_value = float(row[node["feature"]])
                direction = "left" if feature_value <= node["threshold"] else "right"
                node = node[direction]
            predictions[i] = node["value"]
        return predictions

    def loss(self, X, y):
        """Return mean squared prediction error against targets y.

        X follows predict's input requirements. y is a nonempty finite vector
        with one target per row of X.
        """
        predictions = self.predict(X)
        if np.iscomplexobj(y):
            raise ValueError("y must be real-valued.")
        y = np.asarray(y, dtype=np.float64)
        if y.shape != predictions.shape or y.size == 0 or not np.isfinite(y).all():
            raise ValueError("y must be a nonempty finite vector aligned with X.")
        return float(np.mean((predictions - y) ** 2))

## 3. Check model

Reserved for the validation phase.

## 4. References

Breiman, L., Friedman, J.H., Olshen, R.A. and Stone, C.J. (1984)
*Classification and regression trees*. Belmont, CA: Wadsworth.
Available at: [https://doi.org/10.1201/9781315139470](https://doi.org/10.1201/9781315139470).